# 08 — Evaluación final geográfica (protocolo congelado)

Ejecuta la evaluación una sola vez sobre `test_seen` y Bolivia/Paraguay reservados. No entrena ni calibra.

**Precondiciones:** revisión humana de etapa 7 y autorización guardadas en `stage8_preflight_human_reviewed_20261118.json`; protocolo congelado; fuente de código fijada en Drive; root persistente con etapas 1–7 y checkpoints; runtime Colab GPU con Python 3.12.

El notebook no publica imágenes. Las máscaras derivadas quedan en Drive privado; comparte el JSON de métricas y los hashes, no las máscaras/GeoTIFF, mientras la licencia del dataset siga pendiente. Si una compuerta falla, no la cambies ni abras manualmente los datos reservados.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, time
from google.colab import drive
drive.mount("/content/drive")
PERSIST_ROOT = Path("/content/drive/MyDrive/auto-prompt-sam2")
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/auto-prompt-sam2-stage8")
REPO_URL = "https://github.com/roberto-chavez/auto-prompt-sam2.git"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)], check=True)
PIN_FILE = PERSIST_ROOT / "stage8/code_commit.txt"
PIN_FILE.parent.mkdir(parents=True, exist_ok=True)
if PIN_FILE.exists():
    pinned = PIN_FILE.read_text().strip()
    subprocess.run(["git", "fetch", "origin", pinned], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", pinned], cwd=REPO, check=True)
else:
    pinned = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert (REPO / "scripts/run_stage8.py").is_file(), "Este commit no contiene el ejecutor de etapa 8."
if not PIN_FILE.exists():
    PIN_FILE.write_text(pinned + "\n")
print("Code commit pinned in Drive:", pinned)
assert PERSIST_ROOT.is_dir(), "Drive root missing."

## Runtime y entorno fijado
Selecciona GPU T4 (o GPU aprobada por el curso) y Python 3.12/Linux x86_64. No continúes con Python 3.13 ni cambies los pines. Este setup solo instala dependencias; los modelos se cargan offline desde el root de Drive.

In [ ]:
assert sys.version_info[:2] == (3, 12), f"Se requiere Python 3.12; el runtime tiene {sys.version.split()[0]}"
PYTHON = Path("/content/auto-prompt-venv/bin/python")
env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
if not PYTHON.is_file():
    subprocess.run([sys.executable, str(REPO / "scripts/bootstrap_colab.py"), "--venv", str(PYTHON.parent.parent)], cwd=REPO, env=env, check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], cwd=REPO, env=env, check=True)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], text=True))
child_env = dict(env, PYTHONPATH=str(REPO / "src"), AUTO_PROMPT_PERSIST_ROOT=str(PERSIST_ROOT),
                 PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8", CODE_COMMIT=pinned)

## Compuerta y protocolo
Estas celdas leen solo los registros de revisión/configuración. El ejecutor vuelve a comprobar ambas condiciones antes de leer imágenes o etiquetas de conjuntos reservados.

In [ ]:
REVIEW = REPO / "docs/project/evidence/e3/stage8_preflight_human_reviewed_20261118.json"
FREEZE = REPO / "docs/project/evidence/e3/stage8_protocol_freeze_20261118.json"
freeze = json.loads(FREEZE.read_text(encoding="utf-8"))
frozen_sha = freeze["protocol_sha256"]
assert freeze["status"] == "frozen_before_reserved_access"
subprocess.run([str(PYTHON), str(REPO / "scripts/check_stage8_gate.py"), str(REVIEW)], cwd=REPO, env=child_env, check=True)
protocol = json.loads((REPO / "configs/stage8_protocol.json").read_text(encoding="utf-8"))
canonical = json.dumps(protocol, sort_keys=True, separators=(",", ":")).encode()
assert hashlib.sha256(canonical).hexdigest() == frozen_sha, "Protocol hash differs from the pre-access freeze."
print("Protocol frozen SHA-256:", frozen_sha)
print("No reserved images or labels have been read in this cell.")

## Ejecutar evaluación
Primera ejecución: `RESUME=False`. Si Colab se interrumpe, conserva el runtime/log y reanuda con el mismo commit, root, protocolo y GPU mediante `RESUME=True`. No inspecciones resultados reservados para cambiar configuración.

In [ ]:
RESUME = False  # Solo True después de una interrupción técnica del mismo run.
cmd = [str(PYTHON), str(REPO / "scripts/run_stage8.py"), "--root", str(PERSIST_ROOT),
       "--review", str(REVIEW), "--frozen-protocol-sha256", frozen_sha]
if RESUME: cmd.append("--resume")
log_dir = PERSIST_ROOT / "stage8/notebook_logs"
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / f"stage8_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.log"
with log_path.open("x", encoding="utf-8") as log:
    process = subprocess.Popen(cmd, cwd=REPO, env=child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line); log.flush()
    code = process.wait()
if code: raise RuntimeError(f"Stage 8 stopped with code {code}; preserve {log_path} and rerun only with RESUME=True")
print("Persistent log:", log_path)

## Resumen y entrega de evidencia
El reporte JSON no contiene GeoTIFF ni máscaras. Los archivos `predictions/` permanecen en Drive privado. Revisa conteos exactos, `reserved_access`, SHA-256 del protocolo y que ambas comparaciones estén presentes antes de compartir el reporte.

In [ ]:
reports = [p for p in (PERSIST_ROOT / "stage8").glob("*/stage8_report.json")
           if json.loads(p.read_text(encoding="utf-8")).get("run_identity", {}).get("source_commit") == pinned]
assert reports, "No se encontró stage8_report.json; la ejecución no terminó."
REPORT = reports[-1]
summary = json.loads(REPORT.read_text(encoding="utf-8"))
assert summary["status"] == "complete" and summary["protocol_sha256"] == frozen_sha
print("Report:", REPORT)
print("Protocol:", summary["protocol_sha256"])
print("Counts:", {r: {s: d["chips"] for s,d in value["sets"].items()} for r,value in summary["results"].items()})
print("Reserved IDs accessed:", {r: len(ids) for r,ids in summary["reserved_access_by_regime"].items()})
print("SHA-256:", hashlib.sha256(REPORT.read_bytes()).hexdigest())